In [ ]:
!pip install evaluate

Imports

In [ ]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset

import evaluate

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

import torch


Reading Dataset

In [ ]:
df = pd.read_csv("MESC.csv")

# Filter missing values
df = df.dropna(subset=["Utterance", "Emotion"])
df.head()


,Utterance,Speaker,Emotion,Strategy,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime
0,I told you.,Client,sadness,undefined,0,0,1,1,"00:00:54,097","00:00:55,113"
1,Told me what?,Therapist,neutral,Open question,0,1,1,1,"00:00:55,260","00:00:56,263"
2,That you'd be sorry you ever encouraged me to ...,Client,sadness,undefined,0,2,1,1,"00:00:56,651","00:00:59,719"
3,I'm not sorry at all.,Therapist,neutral,Communication Skills,0,3,1,1,"00:00:59,951","00:01:01,237"
4,"You didn't expect it to be like this, I bet.",Client,sadness,undefined,0,4,1,1,"00:01:03,404","00:01:05,206"


In [ ]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["Emotion"])
num_labels = len(le.classes_)
print("Classes:", le.classes_)


Classes: ['anger' 'depression' 'disgust' 'fear' 'joy' 'neutral' 'sadness']


In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df["label"]
)

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)


Tokenizer

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["Utterance"],
        truncation=True,
        max_length=128,
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)


Map:   0%|          | 0/25885 [00:00<?, ? examples/s]

Map:   0%|          | 0/2877 [00:00<?, ? examples/s]

Model

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)


In [ ]:
import transformers
from transformers import TrainingArguments

print("Transformers version:", transformers.__version__)
print("TrainingArguments module:", TrainingArguments.__module__)


Transformers version: 4.57.2
TrainingArguments module: transformers.training_args


In [ ]:
training_args = TrainingArguments(
    output_dir="./emotion_model",
    eval_strategy="epoch",
    save_steps=99999999,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=500,
    logging_steps=50,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


/tmp/ipython-input-943427761.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Training

In [16]:
trainer.train()

                                         [6472/6472 09:21, Epoch 4/4]


Epoch  Training Loss  Validation Loss  Accuracy
  1      1.182800        1.161213      0.608293
  2      1.159700        1.141313      0.616173
  3      0.958100        1.185341      0.623526
  4      0.787300        1.266347      0.626642

TrainOutput(global_step=6472, training_loss=0.8951819379191168, metrics={'train_runtime': 561.7802, 'train_samples_per_second': 184.307, 'train_steps_per_second': 11.521, 'total_flos': 1000062181863000.0, 'train_loss': 0.8951819379191168, 'epoch': 4.0})


Results

In [ ]:
results = trainer.evaluate()
results


{
'eval_loss': 1.3661551475524902,
'eval_accuracy': 0.6214272228015293,
'eval_runtime': 1.912,
'eval_samples_per_second': 1504.683,
'eval_steps_per_second': 94.141,
'epoch': 4.0}
